# 토큰 사용량 확인


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


현재 LangChain의 표준 방식은 공급자 응답을 `AIMessage.usage_metadata`로 정규화해 읽는 것입니다. 여러 호출을 합산할 때는 `UsageMetadataCallbackHandler`를 사용할 수 있습니다. 토큰 수와 실제 청구 금액은 같지 않으며, 가격은 공급자·모델·캐시 정책에 따라 별도로 확인합니다.


In [ ]:
%pip install -qU langchain-core langchain-openai python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    temperature=0,
    stream_usage=True,
)


## 단일 호출


In [ ]:
result = llm.invoke("대한민국의 수도는 어디인가요?")
print(result.text)
print(result.usage_metadata)

usage = result.usage_metadata or {}
print("입력:", usage.get("input_tokens"))
print("출력:", usage.get("output_tokens"))
print("합계:", usage.get("total_tokens"))


## 스트리밍 호출

청크를 더해 최종 메시지를 만들면 마지막 청크에 포함된 사용량도 합쳐집니다.


In [ ]:
full = None
for chunk in llm.stream("RAG를 한 문장으로 설명해 주세요."):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk

print("\n", full.usage_metadata)


## 여러 호출 합산


In [ ]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

usage_callback = UsageMetadataCallbackHandler()
config = {"callbacks": [usage_callback]}

llm.invoke("대한민국의 수도는 어디인가요?", config=config)
llm.invoke("일본의 수도는 어디인가요?", config=config)

# 모델별로 정규화된 사용량이 누적됩니다.
print(usage_callback.usage_metadata)
